# Introduction to Stochastic Regression: an Ensemble of Fits

This tutorial is a follow-on from `intro_to_regression.ipynb`. There, we fit a polynomial to a __single__ set of noisy observations and got a __single__ set of "best fit" parameters - a representation of deterministic groundwater model calibration. But the noise in those observations is random; if we had measured again we would have got slightly different numbers, and therefore slightly different "best fit" parameters.  And our model isnt "perfect", like the order of the polynomial is wrong, then #badtimes

So here we bring in stochastic concepts to see if that can help with the predictive bias (ie wrongness) we saw in the first notebook. We keep the same truth and the same least-squares fit as the original notebook, but we generate __many__ noisy observation sets ("realizations"), fit the model to each one, and look at the spread of the resulting parameters.

Four sliders let you interrogate this:

 * __the observation noise standard deviation__ - how noisy the measurements are;
 * __the number of realizations__ in the ensemble;
 * __the order of the polynomial__ that we fit (the truth is second order);
 * __the correlation length of the observation noise__ - whether the noise is independent from one measurement to the next, or wanders slowly along the x-axis like model error does.

And maybe most importantly, we allow the choice to also estimate the y-intercept of the polynomial - a concept closely related to expanding the parameterization in groundwater modeling.

The point of the exercise: the parameters we estimate are uncertain, and how uncertain they are depends on the data we fit to and on the model we choose to fit.

In [ ]:
import numpy as np
import scipy.optimize as spo
import matplotlib.pyplot as plt
import regression_helper as rh
import ipywidgets as widgets

# the color scheme - change it here and every figure below follows
c_obs = 'red'          # observations, noisy observations and the noise itself
c_fit = 'tab:blue'     # the realizations of the fitted model
c_mean = 'navy'        # the mean of the fitted realizations
c_truth = 'm'          # the truth
c_reals = plt.cm.Reds  # shades of the obs color, for individual noise realizations

## Cook up the same data

As before, `rh.data_cooker()` gives us a "true" second degree polynomial (`poly_func`, with parameters `polypars`) and one noisy set of observations (`y_data`).

This time we also keep the noise-free truth at each x location (`y_true`). That is what we will be adding our own noise to, over and over.

Recall the general form of the polynomial, with $c=0$ so that we have two free parameters, $a$ and $b$:

$y=ax^2 + bx$

In [ ]:
xplot, x, y_data, poly_func, polypars = rh.data_cooker()

# the noise-free "truth" at each x location
y_true = poly_func(x)

# the prediction location, beyond the end of the data. This is the same rule
# the original notebook uses in rh.plot_prediction()
x_pred = x[-1] + 0.21 * (x[-1] - x[0])
y_pred_true = poly_func(x_pred)

print('True parameters are: a={0:.4f}, b={1:.4f}, c={2}'.format(*polypars))
print('{0} observations, x from {1:.2f} to {2:.2f}'.format(len(x), x.min(), x.max()))
print('prediction at x = {0:.2f}, where the true y = {1:.4f}'.format(x_pred, y_pred_true))

Plot the truth

In [ ]:
plt.plot(x,poly_func(x),c=c_truth)

## An ensemble of observations

The noise model starts out as the one `rh.data_cooker()` uses: independent, zero-mean, gaussian noise added to each "true" y value. (`data_cooker()` used a standard deviation of 2.0, which is where the default below comes from.)

`num_reals` is the number of realizations; change it if you want. Each row of `unit_noise` is one realization's worth of standard normal deviates, one per observation. We draw those deviates __once__ and then scale (and, below, correlate) them. That way, when you move a slider, you are seeing the same "measurement campaign" treated differently, rather than an entirely new set of random numbers each time.

In [ ]:
num_reals = 200       # number of realizations in the ensemble
noise_std = 2.0       # the observation noise standard deviation to start with
max_noise_std = 8.0   # the most noise we will ask the slider for
max_corr_len = 50.0    # the longest noise correlation length we will ask for
max_order = 6         # the highest order polynomial we will fit
seed = 42             # so we all see the same thing

rng = np.random.default_rng(seed)
unit_noise = rng.normal(loc=0.0, scale=1.0, size=(num_reals, len(x)))

## Correlated noise

Independent noise is the textbook assumption and it is almost never what we have. In environmental modelling most of the "noise" is not measurement error at all - it is signal coming from the differences between our model and the system that generated the observations - we call this model error, but its also referred to as model discrepancy in some settings. This kind of error does not flip sign between one measurement and the next; its instead typically highly correlated, so nearby measurements are wrong in the same direction.

We can try to mimic that with a correlation function. Here we use an exponential one, with a correlation length $L$:

$\rho(x_i,x_j) = e^{-|x_i - x_j| / L}$

$L=0$ means independent noise (what we had above). Larger $L$ means the noise wanders over a longer stretch of the x-axis. We build the correlation matrix, factor it (Cholesky), and apply the factor to the __same__ unit deviates, so moving the correlation slider re-shapes the noise rather than re-drawing it. Normalizing the rows of the factor keeps the marginal standard deviation of the noise at exactly 1.0, so the noise-standard-deviation slider still means what it says at every correlation length. That matters: we want to change the __structure__ of the noise without changing its __magnitude__.

In [ ]:
# distances between every pair of x locations
x_dist = np.abs(x[:, np.newaxis] - x[np.newaxis, :])

_factor_cache = {}


# Cholesky factor of the exponential correlation matrix, with the rows
# normalized so that the marginal variance stays exactly 1.0
def corr_factor(corr_len):
    if corr_len <= 0:
        return np.eye(len(x))
    if corr_len not in _factor_cache:
        corr = np.exp(-x_dist / corr_len)
        for jitter in [0.0, 1.0e-12, 1.0e-10, 1.0e-8, 1.0e-6]:
            try:
                factor = np.linalg.cholesky(corr + jitter * np.eye(len(x)))
                break
            except np.linalg.LinAlgError:
                continue
        _factor_cache[corr_len] = factor / np.linalg.norm(factor, axis=1, keepdims=True)
    return _factor_cache[corr_len]


def draw_obs_ensemble(noise_std, num_reals=num_reals, corr_len=0.0):
    # one noisy observation vector per realization
    noise = unit_noise[:num_reals, :] @ corr_factor(corr_len).T
    return y_true + noise_std * noise

## What does the noise look like?

Before we fit anything, it is worth looking at the noise on its own. All of these realizations have the same standard deviation - the noise is the same __size__ in every one of them - but not the same __structure__.

That is the interesting part. Measurement error really is close to independent, and it is the small part of the problem. The part that matters in environmental modelling is model error: the things the model cannot represent. Model error does not flip sign from one observation to the next, and the correlated realizations below are a crude stand-in for it.

The figure has three panels: a handful of individual noise realizations; the observations they produce when added to the truth; and the noise correlation measured off the ensemble, plotted against the $e^{-h/L}$ we asked for.

In [ ]:
def plot_noise_reals(noise_std=noise_std, corr_len=0.0, num_show=8):
    noise = draw_obs_ensemble(noise_std, corr_len=corr_len) - y_true
    colors = c_reals(np.linspace(0.45, 0.95, num_show))

    fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(13, 4))

    # a few individual noise realizations
    for i in range(num_show):
        ax0.plot(x, noise[i, :], '-', lw=1, c=colors[i])
    ax0.axhline(0.0, color='k', lw=0.5)
    ax0.set_ylim(-3.5 * max(noise_std, 0.1), 3.5 * max(noise_std, 0.1))
    ax0.set_xlabel('x')
    ax0.set_ylabel('noise')
    ax0.set_title('{0} noise realizations'.format(num_show))
    ax0.grid()

    # the observations they make
    for i in range(num_show):
        ax1.plot(x, y_true + noise[i, :], '.', ms=4, c=colors[i])
    ax1.plot(x, y_true, '-.', lw=2.5, c=c_truth, label='True y',zorder=10)
    pad = max(1.0, 3.0 * noise_std)
    ax1.set_ylim(y_true.min() - pad, y_true.max() + pad)
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.set_title('the observations they make')
    ax1.legend(loc='upper left')
    ax1.grid()

    # the correlation structure, measured off the whole ensemble
    std_noise = (noise - noise.mean(axis=0)) / noise.std(axis=0)
    lags = np.arange(0, len(x) // 2)
    hh = lags * (x[1] - x[0])
    measured = [np.mean(std_noise[:, :len(x) - k] * std_noise[:, k:]) for k in lags]
    ax2.plot(hh, measured, 'o', ms=3, c='0.5', label='measured')
    if corr_len > 0:
        ax2.plot(hh, np.exp(-hh / corr_len), '-', lw=2, c=c_truth,
                 label='$e^{-h/L}$, L = ' + '{0}'.format(corr_len))
    else:
        ax2.plot(hh, np.zeros_like(hh), '-', lw=2, c=c_truth, label='independent')
    ax2.set_ylim(-0.25, 1.05)
    ax2.set_xlabel('separation distance, h')
    ax2.set_ylabel('correlation')
    ax2.set_title('noise correlation')
    ax2.legend(loc='best')
    ax2.grid()

    plt.suptitle('noise std = {0:.2f} (measured {1:.2f})   |   correlation length = {2:.2f}'
                 .format(noise_std, noise.std(), corr_len))
    plt.tight_layout()
    plt.show()

In [ ]:
for corr_len in [0.0, 5.0, 50.0]:
    plot_noise_reals(corr_len=corr_len)

Independent noise (top) is hash: each observation has noise that is unrelated to nearby measurements. Long correlation lengths (bottom) give realizations that wander - each one is a smooth, systematic offset, and it looks exactly like something a model with the wrong parameters would produce.

The sliders below change only the noise - nothing is fit here, so it redraws in a fraction of a second.

In [ ]:
try:
    widgets.interact(plot_noise_reals,
                     noise_std=widgets.FloatSlider(min=0.25, max=max_noise_std, step=0.25,
                                                   value=noise_std, description='noise std'),
                     corr_len=widgets.FloatSlider(min=0.0, max=max_corr_len, step=0.25,
                                                  value=0.0, description='corr len'),
                     num_show=widgets.IntSlider(min=1, max=20, step=1, value=8,
                                                description='num show'));
except Exception as e:
    # no widget machinery available; fall back to a static plot
    print('widgets unavailable ({0}); showing a static plot instead'.format(e))
    plot_noise_reals()

## Fit each realization

Same residual function and same `scipy.optimize.least_squares()` call as the original notebook - the only differences are that we now do it once per realization, and that the polynomial order is an argument.

By default we keep the original notebook's convention of __no constant term__ (the `0` appended to the coefficients): the truth passes through the origin, and leaving the intercept out means the order-2 case is exactly the two-parameter problem of the original notebook. So an order $k$ model has $k$ parameters: the coefficients of $x^k \ldots x^1$.

That convention has a side effect which is worth seeing: every fitted curve is forced through the origin, so the ensemble has exactly __zero__ spread there - a pinch in the fan that is an artifact of what we assumed, not something the data told us. `fit_intercept=True` adds the constant back as a parameter (so an order $k$ model then has $k+1$ of them) and the pinch goes away. There is a checkbox for it on the widget below.

In [ ]:
# build the polynomial, with or without a fitted constant term
def model(pars, fit_intercept=False):
    return np.poly1d(pars if fit_intercept else [*pars, 0])


# residuals = measured values - modelled values
def errfun(pars, x, y_data, fit_intercept=False):
    residuals = y_data - model(pars, fit_intercept)(x)
    return residuals


def fit_one(x, y_data, order=2, fit_intercept=False):
    # the same least-squares fit as the original notebook, at any order
    npar = order + int(fit_intercept)
    sol = spo.least_squares(errfun, np.zeros(npar), args=(x, y_data, fit_intercept))
    return sol.x


def fit_ensemble(noise_std, num_reals=num_reals, order=2, corr_len=0.0,
                 fit_intercept=False, thin=1):
    # thin keeps every thin-th observation; the rest are never seen by the fit
    y_ens = draw_obs_ensemble(noise_std, num_reals, corr_len)[:, ::thin]
    pars = np.array([fit_one(x[::thin], y_obs, order, fit_intercept)
                     for y_obs in y_ens])
    return y_ens, pars


# pull out the two coefficients the truth has: x^2 and x
def quad_coeffs(pars, order, fit_intercept=False):
    i = -3 if fit_intercept else -2
    return (pars[:, i] if order > 1 else None), pars[:, i + 1]


# what each realization predicts at x_pred, out beyond the data
def predictions(pars, fit_intercept=False):
    return np.array([model(p, fit_intercept)(x_pred) for p in pars])

In [ ]:
y_ens, pars = fit_ensemble(noise_std)
ens_a, ens_b = quad_coeffs(pars, 2)
ens_pred = predictions(pars)

print('truth:           a={0:.4f}, b={1:.4f}'.format(*polypars[:2]))
print('ensemble mean:   a={0:.4f}, b={1:.4f}'.format(ens_a.mean(), ens_b.mean()))
print('ensemble stdev:  a={0:.4f}, b={1:.4f}'.format(ens_a.std(), ens_b.std()))
print('prediction at x={0:.2f}: true {1:.3f}, ensemble mean {2:.3f}, stdev {3:.3f}'.format(
    x_pred, y_pred_true, ens_pred.mean(), ens_pred.std()))

Two things worth noting there. The __mean__ of the ensemble of fitted parameters sits essentially on top of the truth - unsurprising, as the noise we added has zero mean and our model has exactly the right form. But the __standard deviation__ is not zero: any one of these fits, including the one we did in the original notebook, is off by some amount that we would not know about if we only ever did the one fit.

Now let's look at the whole ensemble.

## Plot the ensemble

The figure below has four parts:

 * on the left, __the whole observation ensemble__ as a faint red cloud - every realization's noisy y values, all 200 of them at each x location - with one representative realization picked out as solid dots on top. Over that go the ensemble of fitted polynomials (blue), the truth (magenta dash-dot) and the ensemble mean fit (navy). The fitted curves are carried past the end of the data to __the prediction__ at x = 4.26, marked with a dotted vertical line, where the true value is a white-faced circle and the ensemble mean prediction is a navy cross - the same markers, and the same prediction location, as `rh.plot_prediction()` in the original notebook ($x_{pred} = x_{max} + 0.21(x_{max}-x_{min})$);
 * in the middle, histograms of the fitted $x^2$ and $x$ coefficients;
 * on the right, those two coefficients plotted against each other, and below them the histogram of __the ensemble's predictions at x = 4.26__ against the true value.

Plotting the whole cloud, rather than one realization's dots, matters: it is the only way to see on the figure itself that the noise really is at the level the slider says. Any single realization is one draw and can easily look quieter (or rowdier) than the number in the title. The realization drawn with solid dots is chosen as the one whose own noise standard deviation is closest to the requested value, so it is representative rather than lucky.

The parameter axes are deliberately held __fixed__ - they are scaled to the noisiest case, for the current order and correlation length - so that the histograms and the scatter can be compared directly as you change the noise. Otherwise matplotlib would rescale them and everything would look the same. The left panel's y-axis does track the noise level, so watch its numbers.

Changing the order or the correlation length __does__ rescale those axes (there is no single scale that works across all four sliders), so for those two, compare the coefficient standard deviations printed above the figure rather than the width of the histograms. When the noise is correlated, the title also reports how much wider the spread is than it would be with independent noise of the same size.

In [ ]:
_lim_cache = {}


# fix the parameter and prediction axes using the noisiest case we will plot
def par_limits(order, corr_len, fit_intercept=False, thin=1):
    key = (order, corr_len, fit_intercept, thin)
    if key not in _lim_cache:
        wide = fit_ensemble(max_noise_std, order=order, corr_len=corr_len,
                            fit_intercept=fit_intercept, thin=thin)[1]
        lims = []
        for vals in list(quad_coeffs(wide, order, fit_intercept)) + \
                [predictions(wide, fit_intercept)]:
            if vals is None:
                lims.append(None)
                continue
            lims.append((vals.mean() - 3.5 * vals.std(), vals.mean() + 3.5 * vals.std()))
        # make sure the true prediction is always on the prediction axis
        lo, hi = lims[-1]
        pad = 0.05 * (hi - lo)
        lims[-1] = (min(lo, y_pred_true - pad), max(hi, y_pred_true + pad))
        _lim_cache[key] = lims
    return _lim_cache[key]

In [ ]:
def plot_ensemble(noise_std=noise_std, num_reals=num_reals, order=2, corr_len=0.0,
                  fit_intercept=False, thin=1):
    y_ens, pars = fit_ensemble(noise_std, num_reals, order, corr_len, fit_intercept, thin)
    ens_a, ens_b = quad_coeffs(pars, order, fit_intercept)
    ens_pred = predictions(pars, fit_intercept)
    alim, blim, plim = par_limits(order, corr_len, fit_intercept, thin)
    xfit = np.linspace(x[0] - 0.25, x_pred + 0.25, 200)

    # the realization whose own noise level is closest to what was asked for
    noise = y_ens - y_true[::thin]
    irep = np.argmin(np.abs(noise.std(axis=1) - noise_std))

    fig = plt.figure(figsize=(15, 6.5))
    ax = plt.subplot2grid((2, 4), (0, 0), rowspan=2, colspan=2)
    axa = plt.subplot2grid((2, 4), (0, 2))
    axb = plt.subplot2grid((2, 4), (1, 2))
    axs = plt.subplot2grid((2, 4), (0, 3))
    axp = plt.subplot2grid((2, 4), (1, 3))

    # every realization's observations...
    ax.plot(np.tile(x[::thin], (len(y_ens), 1)).ravel(), y_ens.ravel(), '.', ms=3,
            c=c_obs, alpha=0.1, mec='none')
    # ...the ensemble of fitted polynomials, carried out to the prediction...
    for p in pars:
        ax.plot(xfit, model(p, fit_intercept)(xfit), '-', lw=0.5, c=c_fit, alpha=0.25)
    # ...and one representative realization
    ax.plot(x[::thin], y_ens[irep], 'o', ms=4.5, c=c_obs, mec='k', mew=0.4)
    ax.plot(xfit, poly_func(xfit), '-.', lw=2, c=c_truth)
    ax.plot(xfit, model(pars.mean(axis=0), fit_intercept)(xfit), '-', lw=2, c=c_mean)
    # the prediction
    ax.axvline(x_pred, color='k', ls=':', lw=1)
    ax.plot(x_pred, y_pred_true, 'o', ms=11, mfc='w', mec=c_truth, mew=2)
    ax.plot(x_pred, ens_pred.mean(), 'x', ms=11, mew=2, c=c_mean)
    # proxy artists so the faint cloud and fan show up in the legend
    ax.plot([], [], 'o', ms=4, c=c_obs, alpha=0.4, label='Ensemble of noisy y')
    ax.plot([], [], 'o', ms=4.5, c=c_obs, mec='k', mew=0.4, label='One realization')
    ax.plot([], [], '-', lw=1, c=c_fit, label='Ensemble of fits')
    ax.plot([], [], '-.', lw=2, c=c_truth, label='True y')
    ax.plot([], [], '-', lw=2, c=c_mean, label='Ensemble mean fit')
    ax.plot([], [], 'o', ms=9, mfc='w', mec=c_truth, mew=2, label='True prediction')
    ax.plot([], [], 'x', ms=9, mew=2, c=c_mean, label='Mean predicted')
    pad = max(1.0, 3.0 * noise_std)
    ax.set_ylim(min(y_true.min(), y_pred_true) - pad, max(y_true.max(), y_pred_true) + pad)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid()

    # the marginal distribution of the two coefficients the truth has
    for cax, vals, lim, name, truth in zip([axa, axb], [ens_a, ens_b], [alim, blim],
                                           ['$x^2$ coefficient (a)', '$x$ coefficient (b)'],
                                           polypars[:2]):
        cax.set_xlabel(name)
        cax.set_yticks([])
        if vals is None:
            cax.text(0.5, 0.5, 'no $x^2$ term\nin an order 1 model', ha='center',
                     va='center', transform=cax.transAxes, color='0.4')
            cax.set_xticks([])
            continue
        cax.hist(vals, bins=np.linspace(lim[0], lim[1], 40), facecolor=c_fit,
                 edgecolor='none', alpha=0.5)
        cax.axvline(truth, color=c_truth, ls='-.', lw=2, label='Truth')
        cax.axvline(vals.mean(), color=c_mean, lw=2, label='Ensemble mean')
        cax.set_xlim(lim)
        cax.legend(loc='upper left', fontsize=8)

    # and the two against each other
    axs.set_xlabel('$x^2$ coefficient (a)')
    axs.set_ylabel('$x$ coefficient (b)')
    axs.grid()
    if ens_a is None:
        axs.text(0.5, 0.5, 'no $x^2$ term\nin an order 1 model', ha='center',
                 va='center', transform=axs.transAxes, color='0.4')
        axs.set_xticks([])
        axs.set_yticks([])
    else:
        axs.scatter(ens_a, ens_b, marker='.', c=c_fit, alpha=0.5)
        axs.plot(polypars[0], polypars[1], '*', ms=15, c=c_truth, label='Truth')
        axs.plot(ens_a.mean(), ens_b.mean(), 'x', ms=12, mew=2, c=c_mean,
                 label='Ensemble mean')
        axs.set_xlim(alim)
        axs.set_ylim(blim)
        axs.legend(loc='upper left', fontsize=8)

    # the ensemble of predictions, out at x_pred
    axp.hist(ens_pred, bins=np.linspace(plim[0], plim[1], 40), facecolor=c_fit,
             edgecolor='none', alpha=0.5)
    axp.axvline(y_pred_true, color=c_truth, ls='-.', lw=2, label='Truth')
    axp.axvline(ens_pred.mean(), color=c_mean, lw=2, label='Ensemble mean')
    plim = (y_pred_true - 10, y_pred_true + 10)
    axp.set_xlim(plim)
    axp.set_yticks([])
    axp.set_xlabel('prediction at x = {0:.2f}'.format(x_pred))
    axp.legend(loc='upper left', fontsize=8)

    astd = 'n/a' if ens_a is None else '{0:.3f}'.format(ens_a.std())
    inflation = ''
    if corr_len > 0:
        # how much wider is this than the same amount of independent noise?
        indep = fit_ensemble(noise_std, len(y_ens), order, 0.0, fit_intercept, thin)[1]
        inflation = ('   ({0:.1f} times wider than with independent noise of the '
                     'same size)'.format(ens_b.std() / quad_coeffs(indep, order,
                                                                   fit_intercept)[1].std()))
    thinning = '' if thin == 1 else '   |   {0} of {1} observations'.format(
        y_ens.shape[1], len(x))
    plt.suptitle('noise std = {0:.2f} (measured {1:.2f})   |   noise correlation length = {2:.2f}'
                 .format(noise_std, noise.std(), corr_len) +
                 '   |   polynomial order = {0}{1}   |   {2} realizations'.format(
                     order, ' + intercept' if fit_intercept else '', len(y_ens)) + thinning +
                 '\nfitted coefficient standard deviation:  $x^2$ = {0},  $x$ = {1:.3f}'.format(
                     astd, ens_b.std()) + inflation +
                 '\nprediction at x = {0:.2f}:  spread = {1:.3f},  bias = {2:+.3f}'.format(
                     x_pred, ens_pred.std(), ens_pred.mean() - y_pred_true))
    plt.tight_layout()
    plt.show()

## Slider Time

 * __`noise std`__ - the standard deviation of the noise added to the observations. The truth does not change, the model does not change, and the underlying random numbers do not change;
 * __`num reals`__ - how many realizations are in the ensemble. Fewer is handy for picking out individual fits;
 * __`order`__ - the order of the polynomial we fit to each realization. The truth is order 2;
 * __`corr len`__ - the correlation length of the observation noise, in the same units as x (which runs from -3 to 3). Zero is independent noise. The noise magnitude does not change when you move this - only its structure;
 * __`thin`__ - keep only every `thin`-th observation: 1 keeps all 100 of them, 20 keeps 5. Fewer observations means less noise to average away, so the fan of fits and the prediction spread both widen;
 * __`fit intercept`__ - off (the default) forces every fitted curve through the origin, as the original notebook does. On, the constant term becomes a parameter and the pinch at x = 0 opens up.

Every time you move a slider, all of the realizations are re-fit and everything is re-plotted. That takes about a second with 200 realizations, so the sliders are set to update when you let go of them rather than while you drag. If it feels sluggish, turn `num reals` down.

In [ ]:
try:
    widgets.interact(plot_ensemble,
                     noise_std=widgets.FloatSlider(min=0.0, max=max_noise_std, step=0.25,
                                                   value=noise_std, description='noise std',
                                                   continuous_update=False),
                     num_reals=widgets.IntSlider(min=10, max=num_reals, step=10,
                                                 value=num_reals, description='num reals',
                                                 continuous_update=False),
                     order=widgets.IntSlider(min=1, max=max_order, step=1, value=2,
                                             description='order', continuous_update=False),
                     corr_len=widgets.FloatSlider(min=0.0, max=max_corr_len, step=0.25,
                                                  value=0.0, description='corr len',
                                                  continuous_update=False),
                     thin=widgets.IntSlider(min=1, max=20, step=1, value=1,
                                            description='thin',
                                            continuous_update=False),
                     fit_intercept=widgets.Checkbox(value=False,
                                                    description='fit intercept'));
except Exception as e:
    # no widget machinery available; fall back to a static plot
    print('widgets unavailable ({0}); showing a static plot instead'.format(e))
    plot_ensemble(noise_std)

Things to notice as you slide:

 * __noise std__: with very little noise, every realization fits essentially the same parameters - the histograms collapse to a spike and the scatter to a point. With no noise at all, all the realizations are identical and we recover the truth exactly. As the noise grows the fits fan out. The model has not changed - only the data we have is changing. 
 * the fan of fits is narrowest in the middle of the data and widest at the ends. Extrapolating beyond the data is where predictive uncertainty matters.
 * __order__: at order 1 the model cannot bend, so every realization misses the truth in the same way. That is bias arising from model error - the model that generated the observations is different from teh model being asked to fit them.
 * __the prediction__ at x = 4.26 is what we are trying to predict and it requires extrapolation.
 * order 1 is an interesting case. Can you get a good fit with it and capture the prediction?
 * __fit intercept__: with it off, every curve passes exactly through the origin and the prediction uncertainty there is exactly zero.  But with it being fit also, the ability to assimilate highly correlated noisey observations and also make an acceptable prediction, grows.  This is synonamous with adding new types of parameters like recharge, rather than just adding more hydraulic conductivity parameters 
 * __corr len__: as the correlation length grows, the red cloud looks much the same - the noise is the same size - but each individual realization stops looking like scatter and starts looking like a smooth curve. Without any correlation, the noisy obs contain frequency components that cannot be assimilated by the model.  

## Linking to the rest of the tutorials

What we just did has a name, and you will meet it again:

 * fitting a model to many noise realizations is how __Monte Carlo__ analyses of parameter uncertainty work (`part1_12_monte_carlo`);
 * carrying a whole ensemble of parameter sets, each fit to its own noisy realization of the observations, instead of one "calibrated" set, is exactly what PESTPP-IES does (`part1_13_basic_ies` and `part2_06_ies`). It is doing this same experiment, with a groundwater model in place of our polynomial;
 * choosing what noise to put on observations, and how it relates to the weights we give them, is the subject of `part2_02_obs_and_weights` (`freyberg_obs_and_weights.ipynb` and `weights_vs_noise.ipynb`). The correlated noise slider here is a stand-in for model error, which is what dominates the "noise" in any real environmental model;
 * the reason noisier data gives less certain parameters, and the machinery for saying so without brute force, is the subject of the Bayes and FOSM tutorials (`part0_01_intro_to_bayes`, `part1_10_intro_to_fosm`).

The take away from this notebook: __a single calibrated parameter set is one draw from a distribution__. The width of that distribution is set by the noise in the data we calibrated to, by the structure of that noise, and by how many parameters we chose to fit. Reporting the one fit without the spread around it hides most of what we need to know to make a decision with the model.